# Regularized Matrix Factorization

This collaborative filtering technique predicts user preferences by decomposing a spare user-item interaction matrix into lower-dimenssional, dense user and item latent factor matrices. It adds regularization terms to the loss function to penalize large factor values, preventing overfitting and improving generalization on unseen data.

### 1. Load Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from math import sqrt


In [2]:
df = pd.read_csv("../data/ratings_small.csv")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


### 2. Clean data, split into train/test (based on timestamp)

We want to drop sparse users, then base the train/test split on the timestamp a user has watched the movie. 

In [3]:
# drop users with less than 5 ratings
counts = df["userId"].value_counts()
df = df[df['userId'].isin(counts[counts >= 5].index)].copy()

# split into train and test sets -- 80% train, 20% test
df.sort_values(['userId', 'timestamp'], inplace=True)

train, test = [], []
for userID, group in df.groupby('userId'):
    cut = int(len(group) * 0.8)
    train.append(group.iloc[:cut])
    test.append(group.iloc[cut:])
train_set = pd.concat(train).reset_index(drop=True)
test_set = pd.concat(test).reset_index(drop=True)

print(f"\nTrain: {len(train_set):,} | Test: {len(test_set):,}")



Train: 79,748 | Test: 20,256


### 3. Index encoding

Since the movie/user IDs are arbitrary numbers, we want to map them to contiguous integer indices after filtering through each entry

In [4]:
#get all unique users and movies
all_users = df['userId'].unique()
all_movies = df['movieId'].unique()

# create mappings from raw IDs to indices
user2index = {u: i for i, u in enumerate(all_users)}
movie2index = {m: i for i, m in enumerate(all_movies)}

# number of unique users and movies
n_users = len(all_users)
n_movies = len(all_movies)
print(f"n_users={n_users}  n_movies={n_movies}")

# encode user and movie IDs in the train and test sets
def encode(df_split):
    mask = df_split['movieId'].isin(movie2index)
    d = df_split[mask]
    u = d['userId'].map(user2index).values
    m = d['movieId'].map(movie2index).values
    r = d['rating'].values.astype(np.float32)
    return u, m, r

u_tr, m_tr, r_tr = encode(train_set)
u_te, m_te, r_te = encode(test_set)
print(f"Train triples: {len(r_tr):,}  |  Test triples: {len(r_te):,}")


n_users=671  n_movies=9066
Train triples: 79,748  |  Test triples: 20,256


### 4. Model
Regularized matrix factorization trained with SGD

**Objective**
$$
\min_{P,Q,b_u,b_i}\sum_{(u,i)\in\mathcal{K}}\!\left(r_{ui} - \mu - b_u - b_i - p_u^\top q_i\right)^2
+ \lambda\!\left(\|p_u\|^2+\|q_i\|^2+b_u^2+b_i^2\right)
$$

| Symbol | Meaning |
|--------|--------|
| $P\in\mathbb{R}^{n_u\times k}$ | User latent matrix |
| $Q\in\mathbb{R}^{n_i\times k}$ | Item latent matrix |
| $b_u, b_i$ | User / item bias |
| $\mu$ | Global mean |
| $\lambda$ | L2 regularization coefficient |


In [5]:
class MatrixFactorization:
    def __init__(self, n_users, n_items, n_factors=20, lr=0.005, reg=0.02, n_epochs=20, seed=42):
        self.n_factors = n_factors # latent factors
        self.lr = lr # learning rate
        self.reg = reg # regularization strength
        self.n_epochs = n_epochs # training epochs

        rng = np.random.default_rng(seed)
        scale = 0.1
        self.P  = rng.normal(0, scale, (n_users, n_factors)).astype(np.float32)   # user factors
        self.Q  = rng.normal(0, scale, (n_items, n_factors)).astype(np.float32)   # item factors
        self.bu = np.zeros(n_users, dtype=np.float32)   # user biases
        self.bi = np.zeros(n_items, dtype=np.float32)   # item biases
        self.mu = 0.0                                    # global mean (set at fit time)


    # prediction -- global mean rating + user bias + item bias + dot product of user and item factors
    def predict(self, u, m):
        return self.mu + self.bu[u] + self.bi[m] + np.sum(self.P[u] * self.Q[m], axis=1)
    
    # training 
    def fit(self, u_tr, m_tr, r_tr, u_te=None, m_te=None, r_te=None, verbose=True):
        # set global mean to the average rating in the training set
        self.mu = r_tr.mean()
        n = len(r_tr)
        idx = np.arange(n)

        # track RMSE history for train and test sets
        self.train_rmse_hist = []
        self.test_rmse_hist  = []

        for epoch in range(1, self.n_epochs + 1):
            np.random.shuffle(idx)      # in-place shuffle for SGD

            # loop over training triples and update parameters
            for i in idx:
                u, m, r = u_tr[i], m_tr[i], r_tr[i]
                err = r - (self.mu + self.bu[u] + self.bi[m] + self.P[u] @ self.Q[m])

                # gradient step
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[m] += self.lr * (err - self.reg * self.bi[m])
                pu_old      = self.P[u].copy()
                self.P[u]  += self.lr * (err * self.Q[m] - self.reg * self.P[u])
                self.Q[m]  += self.lr * (err * pu_old    - self.reg * self.Q[m])

            # epoch metrics
            tr_rmse = sqrt(mean_squared_error(r_tr, self.predict(u_tr, m_tr)))
            self.train_rmse_hist.append(tr_rmse)

            if u_te is not None:
                te_rmse = sqrt(mean_squared_error(r_te, self.predict(u_te, m_te)))
                self.test_rmse_hist.append(te_rmse)

            if verbose and epoch % 5 == 0:
                msg = f"Epoch {epoch:>3}/{self.n_epochs}  train RMSE={tr_rmse:.4f}"
                if u_te is not None:
                    msg += f"  test RMSE={te_rmse:.4f}"
                print(msg)

        return self


### 5. Train

In [6]:
model = MatrixFactorization(
    n_users=n_users,
    n_items=n_movies,
    n_factors=20,
    lr=0.005,
    reg=0.02,
    n_epochs=30,
)

model.fit(u_tr, m_tr, r_tr, u_te, m_te, r_te)

Epoch   5/30  train RMSE=0.8800  test RMSE=0.9296
Epoch  10/30  train RMSE=0.8494  test RMSE=0.9187
Epoch  15/30  train RMSE=0.8225  test RMSE=0.9146
Epoch  20/30  train RMSE=0.7879  test RMSE=0.9128
Epoch  25/30  train RMSE=0.7451  test RMSE=0.9129
Epoch  30/30  train RMSE=0.7004  test RMSE=0.9149


### 6. Evaluation

In [7]:
def rmse(true, pred): return sqrt(mean_squared_error(true, pred))
def mae(true, pred):  return np.mean(np.abs(true - pred))

train_preds = model.predict(u_tr, m_tr)
test_preds  = model.predict(u_te, m_te)

# Clip predictions to valid rating range
rating_min, rating_max = df['rating'].min(), df['rating'].max()
test_preds_clipped = np.clip(test_preds, rating_min, rating_max)

print("=" * 40)
print(f"Train RMSE : {rmse(r_tr, train_preds):.4f}")
print(f"Test  RMSE : {rmse(r_te, test_preds):.4f}  (raw)")
print(f"Test  RMSE : {rmse(r_te, test_preds_clipped):.4f}  (clipped to [{rating_min},{rating_max}])")
print(f"Test  MAE  : {mae(r_te, test_preds_clipped):.4f}")
print("=" * 40)

Train RMSE : 0.7004
Test  RMSE : 0.9149  (raw)
Test  RMSE : 0.9145  (clipped to [0.5,5.0])
Test  MAE  : 0.6988


### 7. Adjust hyperparams

### 8. Top-N recommendations

In [8]:
# map movieID to their titles
movies = pd.read_csv("../data/movies_metadata.csv", low_memory=False)
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

links = pd.read_csv("../data/links.csv")
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
links = links.dropna(subset=['tmdbId'])
links['movieId'] = links['movieId'].astype(int)
links['tmdbId']  = links['tmdbId'].astype(int)

tmdb2title = dict(zip(movies['id'], movies['original_title']))
movielens2title = {
    row.movieId: tmdb2title.get(row.tmdbId, None)
    for row in links.itertuples()
}


def recommend(model, user_raw_id, n=10, seen_movie_ids=None):
    # get user index from raw ID, all movies, and predictions for all movies
    u_idx = user2index[user_raw_id]
    all_m = np.arange(n_movies)
    preds = model.predict(np.full(n_movies, u_idx), all_m)

    # exclude seen movies by setting their predicted scores to -inf
    if seen_movie_ids:
        seen_idx = [movie2index[m] for m in seen_movie_ids if m in movie2index]
        preds[seen_idx] = -np.inf

    # sort predictions and get the top N movie indices, then map back to raw movie IDs and return with predicted scores
    top_idx = np.argsort(preds)[::-1][:n]
    idx2movie = {v: k for k, v in movie2index.items()}

    return [(movielens2title.get(idx2movie[i], "Unknown Title"), round(preds[i], 3)) for i in top_idx]

sample_user = all_users[0]
seen = set(train_set[train_set['userId'] == sample_user]['movieId'])

print(f"Top-10 recommendations for user {sample_user}:")
for rank, (title, score) in enumerate(recommend(model, sample_user, n=10, seen_movie_ids=seen), 1):
    print(f"  {rank:>2}. Movie: {title} | Predicted rating: {score}")



Top-10 recommendations for user 1:
   1. Movie: Happiness | Predicted rating: 3.6989998817443848
   2. Movie: Trois couleurs : Bleu | Predicted rating: 3.6579999923706055
   3. Movie: The African Queen | Predicted rating: 3.63700008392334
   4. Movie: The Shawshank Redemption | Predicted rating: 3.635999917984009
   5. Movie: On the Waterfront | Predicted rating: 3.562000036239624
   6. Movie: The Godfather: Part II | Predicted rating: 3.555999994277954
   7. Movie: The Conversation | Predicted rating: 3.5380001068115234
   8. Movie: 12 Angry Men | Predicted rating: 3.5339999198913574
   9. Movie: Amores perros | Predicted rating: 3.5160000324249268
  10. Movie: Trois couleurs : Rouge | Predicted rating: 3.5139999389648438


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from math import sqrt
import random


def score_candidates(model, user_raw_id, candidate_movie_ids):
    u_idx = user2index[user_raw_id]

    movie_indices = [movie2index[m] for m in candidate_movie_ids if m in movie2index]
    valid_movies = [m for m in candidate_movie_ids if m in movie2index]

    if len(movie_indices) == 0:
        return [], np.array([])

    u_vec = np.full(len(movie_indices), u_idx)
    scores = model.predict(u_vec, np.array(movie_indices))

    return valid_movies, scores


def sample_candidates_for_user(user_id, ratings_df, all_movie_ids, n_candidates=25, rng=None):
    """Sample one positive + negatives once for a user."""
    user_rated = ratings_df[ratings_df['userId'] == user_id]['movieId'].unique()
    not_watched = list(all_movie_ids - set(user_rated))

    if len(user_rated) == 0 or len(not_watched) < n_candidates - 1:
        return None

    if rng is None:
        rng = random

    watched_movie = rng.choice(list(user_rated))
    negative_sample = rng.sample(not_watched, n_candidates - 1)
    candidates = [watched_movie] + negative_sample
    rng.shuffle(candidates)

    return {
        'watched_movie': watched_movie,
        'candidates': candidates
    }


def build_shared_candidate_sets(users, ratings_df, n_candidates=25, seed=42):
    """Build one shared candidate set per user for fair model-to-model comparison."""
    rng = random.Random(seed)
    all_movie_ids = set(ratings_df['movieId'].unique())

    shared = {}
    for user_id in users:
        sampled = sample_candidates_for_user(
            user_id=user_id,
            ratings_df=ratings_df,
            all_movie_ids=all_movie_ids,
            n_candidates=n_candidates,
            rng=rng
        )
        if sampled is not None:
            shared[user_id] = sampled

    return shared


def evaluate_model(model, users, ratings_df, k=3, n_candidates=25, seed=42, shared_candidates=None):
    random.seed(seed)
    np.random.seed(seed)

    all_movie_ids = set(ratings_df['movieId'].unique())

    hits, ndcgs = [], []

    for user_id in users:
        if shared_candidates is not None:
            sampled = shared_candidates.get(user_id)
            if sampled is None:
                continue
            watched_movie = sampled['watched_movie']
            candidates = sampled['candidates']
        else:
            sampled = sample_candidates_for_user(
                user_id=user_id,
                ratings_df=ratings_df,
                all_movie_ids=all_movie_ids,
                n_candidates=n_candidates
            )
            if sampled is None:
                continue
            watched_movie = sampled['watched_movie']
            candidates = sampled['candidates']

        valid_movies, scores = score_candidates(model, user_id, candidates)
        if len(scores) == 0:
            continue

        ranked_idx = np.argsort(scores)[::-1]
        ranked_movies = [valid_movies[i] for i in ranked_idx]

        hits.append(int(watched_movie in ranked_movies[:k]))

        if watched_movie in ranked_movies:
            rank = ranked_movies.index(watched_movie)
            ndcg = 1 / np.log2(rank + 2)
        else:
            ndcg = 0

        ndcgs.append(ndcg)

    return {
        'hit_rate': np.mean(hits) if hits else 0.0,
        'ndcg': np.mean(ndcgs) if ndcgs else 0.0,
        'n_users': len(hits),
    }


test_users = test_set['userId'].unique()
ratings = pd.concat([train_set, test_set]).reset_index(drop=True)

# Shared user-level candidate pools for consistent comparison across models.
shared_eval_candidates = build_shared_candidate_sets(
    users=test_users,
    ratings_df=ratings,
    n_candidates=25,
    seed=42
)

results_mf = evaluate_model(
    model=model,
    users=test_users,
    ratings_df=ratings,
    shared_candidates=shared_eval_candidates
)
print("=" * 40)
print("Matrix Factorization Results:")
print(f"Hit Rate@3: {results_mf['hit_rate']:.4f}  |  NDCG@3: {results_mf['ndcg']:.4f}  |  Evaluated Users: {results_mf['n_users']}")
print(f"Shared candidate sets built for users: {len(shared_eval_candidates)}")

Matrix Factorization Results:
Hit Rate@3: 0.3696  |  NDCG@3: 0.4669  |  Evaluated Users: 671
Shared candidate sets built for users: 671


In [10]:
import ast

def get_top3_table(user_id, model, train_df, movies, links,
                   user2index, movie2index, candidate_movie_ids, n=3):

    # --- Build metadata (titles, genres, actors) ---
    movies = movies.copy()
    movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
    movies = movies.dropna(subset=['id'])
    movies['id'] = movies['id'].astype(int)

    def parse_genres(x):
        try:
            return ", ".join([g['name'] for g in ast.literal_eval(x)])
        except:
            return "Unknown"

    def parse_cast(x):
        try:
            return ", ".join([c['name'] for c in ast.literal_eval(x)[:3]])
        except:
            return "Unknown"

    movies['genres'] = movies['genres'].apply(parse_genres)
    movies['actors'] = movies['cast'].apply(parse_cast)

    # --- Build ID mappings ---
    links = links.copy()
    links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
    links = links.dropna(subset=['tmdbId'])
    links['movieId'] = links['movieId'].astype(int)
    links['tmdbId'] = links['tmdbId'].astype(int)

    tmdb2meta = {
        row.id: (row.original_title, row.genres, row.actors)
        for row in movies.itertuples()
    }

    movielens2meta = {
        row.movieId: tmdb2meta.get(row.tmdbId, ("Unknown", "Unknown", "Unknown"))
        for row in links.itertuples()
    }

    u_idx = user2index[user_id]
    valid_movies = [m for m in candidate_movie_ids if m in movie2index]
    movie_indices = [movie2index[m] for m in valid_movies]

    if len(movie_indices) == 0:
        return pd.DataFrame(columns=["Movie", "Genres", "Actors", "Predicted Rating", "User Rating"])

    preds = model.predict(np.full(len(movie_indices), u_idx), np.array(movie_indices))

    # --- Top-N from shared candidate set ---
    top_idx = np.argsort(preds)[::-1][:n]

    rows = []
    for i in top_idx:
        movie_id = valid_movies[i]
        title, genres, actors = movielens2meta.get(movie_id, ("Unknown", "Unknown", "Unknown"))

        rating = train_df[
            (train_df['userId'] == user_id) &
            (train_df['movieId'] == movie_id)
        ]['rating']
        rating = rating.values[0] if len(rating) > 0 else "Not Rated"

        rows.append({
            "Movie": title,
            "Genres": genres,
            "Actors": actors,
            "Predicted Rating": round(float(preds[i]), 3),
            "User Rating": rating
        })

    return pd.DataFrame(rows)


# === Run with shared candidates ===
sample_user = all_users[5]

# Ensure `movies` has a `cast` column (from credits metadata) to avoid KeyError
if 'cast' not in movies.columns:
    credits = pd.read_csv("../data/credits.csv", low_memory=False)
    credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
    credits = credits.dropna(subset=['id'])
    credits['id'] = credits['id'].astype(int)

    movies = movies.copy()
    movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
    movies = movies.dropna(subset=['id'])
    movies['id'] = movies['id'].astype(int)

    movies = movies.merge(credits[['id', 'cast']], on='id', how='left')

movies['cast'] = movies['cast'].fillna('[]')

ratings = pd.concat([train_set, test_set]).reset_index(drop=True)
demo_candidates = build_shared_candidate_sets(
    users=[sample_user],
    ratings_df=ratings,
    n_candidates=25,
    seed=42
)

if sample_user in demo_candidates:
    watched_movie = demo_candidates[sample_user]['watched_movie']
    candidate_movie_ids = demo_candidates[sample_user]['candidates']
    print(f"Candidate pool size: {len(candidate_movie_ids)}")

    table = get_top3_table(
        user_id=sample_user,
        model=model,
        train_df=train_set,
        movies=movies,
        links=links,
        user2index=user2index,
        movie2index=movie2index,
        candidate_movie_ids=candidate_movie_ids,
        n=3
    )

    print(table)
else:
    print(f"User {sample_user} does not have enough unrated movies to build shared candidates.")

Candidate pool size: 25
                    Movie                        Genres  \
0    Inglourious Basterds  Drama, Action, Thriller, War   
1       Kramer vs. Kramer                         Drama   
2  Sweet Smell of Success                         Drama   

                                         Actors  Predicted Rating User Rating  
0   Brad Pitt, Mélanie Laurent, Christoph Waltz             3.819   Not Rated  
1  Dustin Hoffman, Meryl Streep, Jane Alexander             3.496   Not Rated  
2   Burt Lancaster, Tony Curtis, Susan Harrison             3.412   Not Rated  
